## Inside the encoder: tokens, pooling, and model options

`00_preparation/01_encodings` runs whole datasets through UNI2-h / Virchow2 using the `embed_images` helper in `vfm_encoders.py` -- one function call, one pooled vector per patch. That's the right tool once you trust the pipeline, but it hides everything interesting: what actually comes out of the model before pooling, what the CLS/register/patch tokens are, and why the two encoders' recommended pooling differs.

This notebook opens that box by hand, on a handful of PathMNIST patches -- no full dataset, no saved files, just the model objects and their raw tensors. The next notebook (`02_encode_it_yourself.ipynb`) then has you rebuild the `00_preparation/01_encodings` pipeline yourself at a tiny scale.

In [ ]:
# Install dependencies, then restart the kernel. This is only needed once per environment.
# !pip install -U -r /home/shared/helper/requirements.txt

In [ ]:
import sys

sys.path.append("/home/shared/helper/")
import os

os.environ["HF_HOME"] = "/home/shared/.cache/huggingface"
import torch
from nbhelper import (
    np,
    plt,
)
from PIL import Image

from vfm_encoders import embed_images, load_uni2, load_virchow2  # noqa: E402

### 0. Setup

Both encoders are gated on Hugging Face -- request access on the [UNI2-h](https://huggingface.co/MahmoodLab/UNI2-h) and [Virchow2](https://huggingface.co/paige-ai/Virchow2) model pages with the account you authenticate as below, then set `HF_TOKEN` in the environment (or leave it unset to get an interactive login prompt).

In [ ]:
import os
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN"))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print(
        "WARNING: no GPU found -- these are large ViT-H encoders, CPU is only "
        "fine for a quick smoke test on a handful of patches."
    )

### 1. A handful of patches

Just 8 PathMNIST patches -- plenty to inspect tensors with, not a dataset to embed.

In [ ]:
USE_SUBSET = True

pathmnist_base_dir = "/home/shared/data/pathmnist/"
pathmnist_filename = "pathmnist_224_subset1000.npz" if USE_SUBSET else "pathmnist_224.npz"
pathmnist_path = pathmnist_base_dir + pathmnist_filename

LABEL_NAMES = {
    0: "adipose",
    1: "background",
    2: "debris",
    3: "lymphocytes",
    4: "mucus",
    5: "smooth muscle",
    6: "normal colon mucosa",
    7: "cancer-associated stroma",
    8: "colorectal adenocarcinoma epithelium",
}

data = np.load(pathmnist_path)
n_samples = 12
images = [Image.fromarray(img).convert("RGB") for img in data["train_images"][:n_samples]]
labels = data["train_labels"].reshape(-1)[:n_samples]

fig, axs = plt.subplots(nrows=3, ncols=n_samples // 3, figsize=(12, 12))
for i, (img, label) in enumerate(zip(images, labels)):
    axs.flat[i].imshow(img)
    axs.flat[i].set_title(f"Label: {LABEL_NAMES[label]}")
    axs.flat[i].axis("off")

### 2. Load UNI2-h and look at its raw output

`load_uni2` builds the model with `num_classes=0`, which makes `model(batch)` return an already-pooled CLS embedding -- convenient for `embed_images`, but it means the *sequence* of tokens the ViT actually produces is hidden. `model.forward_features(batch)` gives you that sequence back, before pooling.

In [ ]:
uni2 = load_uni2(device)

batch = uni2.transform(images[0]).unsqueeze(0).to(device)
with torch.inference_mode():
    tokens = uni2.model.forward_features(batch)

print("input batch:", tuple(batch.shape))
print("token sequence:", tuple(tokens.shape))

UNI2-h is built with `reg_tokens=8` and `no_embed_class=True` -- so the sequence is `[CLS, reg_0, ..., reg_7, patch_0, ..., patch_{N-1}]`: 1 CLS + 8 register tokens, then the spatial patch grid. `patch_size=14` and a 224px input give a `224/14 = 16` patch grid per side, so `16 * 16 = 256` patch tokens -- check that `1 + 8 + 256` matches the sequence length you just printed.

In [ ]:
patch_size = uni2.model.patch_embed.patch_size[0]
grid_h = batch.shape[-2] // patch_size
grid_w = batch.shape[-1] // patch_size
n_prefix = 1 + 8  # CLS + 8 register tokens

print(
    f"patch_size={patch_size}  grid={grid_h}x{grid_w}  "
    f"expected total={n_prefix + grid_h * grid_w}  actual={tokens.shape[1]}"
)

### PCA of encodings on an image

In [ ]:
tokens_featuremap = tokens[:, n_prefix:].reshape(batch.shape[0], grid_h, grid_w, -1).cpu().numpy()

# reduce tokens_featuremap with PCA to 3 dims
from sklearn.decomposition import PCA

pca = PCA(n_components=3)

tokens_pca = pca.fit_transform(tokens_featuremap.reshape(-1, tokens_featuremap.shape[-1])).reshape(
    tokens_featuremap.shape[0], tokens_featuremap.shape[1], tokens_featuremap.shape[2], 3
)

# minmax scale tokens_pca between 0 and 1
tokens_pca = (tokens_pca - tokens_pca.min()) / (tokens_pca.max() - tokens_pca.min())

In [ ]:
# compare to original image
fig, axs = plt.subplots(ncols=2)

axs[0].imshow(images[0])
axs[1].imshow(tokens_pca[0])

### Exercise: do the same for Virchow2

Load Virchow2 with `load_virchow2`. Unlike UNI2-h, its model is *not* built with `num_classes=0` -- `virchow2.model(batch)` already returns the full `(B, N, D)` token sequence directly, no `forward_features` needed. Run one image through it and print the raw shape.

Virchow2 uses **4** register tokens (check the module docstring in `vfm_encoders.py` if you want the source). Using the same patch-grid math as above, does `1 + 4 + grid_h * grid_w` match the sequence length you see? Virchow2's patch size and default image size are the same as UNI2-h's -- confirm that from `virchow2.model.patch_embed.patch_size` rather than assuming it.

### 3. Pooling strategies

Two models, two different answers to "how do I turn a token sequence into one embedding vector":

- **UNI2-h**: `num_classes=0` already pools to the CLS token inside the model, so `Encoder.pool` for UNI2-h is just the identity -- there's nothing left to do.
- **Virchow2**: the model card's recommended embedding is CLS *concatenated* with the mean of the patch tokens (registers excluded) -- `1280 + 1280 = 2560`-d. `load_virchow2`'s `pool` function implements exactly this.

Reconstruct Virchow2's recommended embedding by hand for one image, and check it matches what `embed_images` (which calls `pool` internally) gives you.

In [ ]:
virchow2 = load_virchow2(device)
batch = virchow2.transform(images[0]).unsqueeze(0).to(device)

with torch.inference_mode():
    tokens = virchow2.model(batch)
    class_token = tokens[:, 0]
    patch_tokens = tokens[:, 5:]  # tokens 1:4 are registers
    manual_embedding = torch.cat([class_token, patch_tokens.mean(dim=1)], dim=-1)


helper_embedding = embed_images(virchow2, [images[0]], device=device, show_progress=False)
print(
    "manual vs. embed_images match:",
    np.allclose(manual_embedding.cpu().numpy(), helper_embedding, atol=1e-4),
)

### Exercise: try a different pooling strategy

The concat-pooling above isn't the only option. For Virchow2, implement two alternatives on your 8 patches:

1. **CLS-only** -- just `tokens[:, 0]` (1280-d).
2. **Mean-patch-only** -- drop CLS entirely, mean-pool `tokens[:, 5:]` (1280-d).

For each pooling strategy, compute cosine similarity between two patches you know share a label (use `labels` from the setup cell) and between two patches with different labels. Does any one strategy separate same-label from different-label pairs more clearly than the others, even just eyeballing 2-3 pairs? (`sklearn.metrics.pairwise.cosine_similarity` or a manual dot-product-over-norms both work.)

### 4. Model config knobs

`load_uni2` and `load_virchow2` pass different `timm_kwargs` into `timm.create_model`. A few worth knowing:

- **`patch_size=14`** -- shared by both. Smaller patches -> more tokens per image -> finer spatial resolution, at the cost of a longer sequence.
- **`reg_tokens`** -- 8 for UNI2-h, 4 for Virchow2. Register tokens are a ViT training trick (see the [Vision Transformers Need Registers](https://arxiv.org/abs/2309.16588) paper): extra tokens the model can dump high-norm "junk" activations into, which otherwise pollute the CLS/patch tokens' attention maps.
- **`dynamic_img_size=True`** (UNI2-h only) -- lets the model accept input sizes other than the one it was trained at, by interpolating position embeddings on the fly instead of requiring a fixed 224px input.
- **`mlp_layer=SwiGLUPacked`, `act_layer=SiLU`** -- shared by both; an architectural choice (SwiGLU-style MLP blocks) independent of any of the above.

See the model cards linked in `vfm_encoders.py`'s module docstring for the full architecture and training details.

### Exercise: what happens with a larger input image?

UNI2-h sets `dynamic_img_size=True`. Take one PathMNIST patch, resize it to 448x448 (you'll need to build your own resize+normalize transform, or edit the tensor after `uni2.transform` -- don't just call `uni2.transform` again on a resized image if its own resize step would undo your resize), and run it through `uni2.model.forward_features` directly.

How many patch tokens come out? Does it match `(448 / patch_size) ** 2`? Now try the same 448x448 input on `virchow2.model` -- it does *not* set `dynamic_img_size`. You don't need to make it work; just observe what happens (an error, or a shape you didn't expect) and note why, given what `dynamic_img_size` does.

### Takeaways

- UNI2-h and Virchow2 disagree on *how many* prefix tokens to drop (9 vs. 5) and *how* to pool the rest (identity-via-`num_classes=0` vs. CLS+mean-patch concat) -- both choices are baked into `vfm_encoders.py` so `embed_images` can treat both encoders uniformly, but the choices themselves aren't interchangeable across models.
- Register tokens exist so CLS/patch tokens don't have to double as a dumping ground for whatever the model needs to compute internally -- that's why they're dropped rather than pooled in.
- `dynamic_img_size` is a per-model config choice, not a general ViT property -- always check before assuming an encoder handles a non-default input size gracefully.

Next: `02_encode_it_yourself.ipynb` has you rebuild a tiny version of the `02_encodings` pipeline -- load patches, call `embed_images` yourself, save, and sanity-check the result -- before running the full per-dataset notebooks.